In [1]:
library(terra)
library(ggplot2)
library(sf)
library(sp)
#library(rgdal)
library(hexbin)
library(stringr)
library(tidyverse)
library(aws.signature)
library(aws.s3)

Sys.setenv("AWS_DEFAULT_REGION" = "us-west-2") 

terra 1.7.78

Linking to GEOS 3.12.1, GDAL 3.8.5, PROJ 9.4.0; sf_use_s2() is TRUE

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ tidyr::extract() masks terra::extract()
✖ dplyr::filter()  masks stats::filter()
✖ dplyr::lag()     masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
boreal_tile <- st_read('/projects/my-public-bucket/databank/boreal_tiles_v004_model_ready.gpkg')
#boreal_tile <- spTransform(boreal_tile,"+proj=longlat +datum=WGS84 +no_defs")
metrics_list <- c("RH098","RH095","RH090","RH085","RH080","CC_gte_01p00m","CC_gte_01p37m","CC_gte_02p00m","ZG_mean")
lvis_path <- read.csv('/projects/shared-buckets/tonyumd/lvis_metrics_s3list.csv')
ht_tindex_master <- read.csv('/projects/shared-buckets/montesano/DPS_tile_lists/BOREAL_MAP/dev_v1.5/Ht_H30_2020/full_run_no_uncert/HT_tindex_master.csv')

Reading layer `boreal_tiles_v004_model_ready' from data source 
  `/projects/my-public-bucket/databank/boreal_tiles_v004_model_ready.gpkg' 
  using driver `GPKG'
Simple feature collection with 4956 features and 5 fields
Geometry type: POLYGON
Dimension:     XY
Bounding box:  xmin: -6111478 ymin: 1233304 xmax: 6308522 ymax: 10323300
Projected CRS: unnamed


In [3]:
RH098_list <- lvis_path[grep(paste0("RH098"), lvis_path$X0),]
RH098_list[1]
substring(RH098_list[2007],first = 78,last = 104)

[1] "ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_056233_RH098_mean_30m.tif"

[1] "ABoVE2019_0723_R2003_068589"

In [4]:
gridded_lvis_rh098_name_list <- list.files('/projects/LVIS_F_Gridded/',pattern = "RH098")
gridded_lvis_rh098_name_list

character(0)

In [122]:
intersect(grep(paste0("ZG_mean"), lvis_path$X0), grep(paste0(substring(RH098_list[100],first = 78,last = 104)), lvis_path$X0))
lvis_path$X0[4598]

[1] 4598

[1] "ornl-cumulus-prod-protected/above/ABoVE_LVIS_VegetationStructure/data/LVISF3_ABoVE2017_0629_R1803_094112_ZG_mean_30m.tif"

In [124]:
ht_tindex_master

index,s3_path,local_path,file,tile_num
<int>,<chr>,<chr>,<chr>,<int>
0,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/11/50/073886/boreal_ht_2020_202412131734106305_0000013.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/11/50/073886/boreal_ht_2020_202412131734106305_0000013.tif,boreal_ht_2020_202412131734106305_0000013.tif,13
1,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/11/055272/boreal_ht_2020_202412131734106325_0000065.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/11/055272/boreal_ht_2020_202412131734106325_0000065.tif,boreal_ht_2020_202412131734106325_0000065.tif,65
2,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/18/416348/boreal_ht_2020_202412131734106332_0000106.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/18/416348/boreal_ht_2020_202412131734106332_0000106.tif,boreal_ht_2020_202412131734106332_0000106.tif,106
3,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/23/291547/boreal_ht_2020_202412131734106336_0000021.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/23/291547/boreal_ht_2020_202412131734106336_0000021.tif,boreal_ht_2020_202412131734106336_0000021.tif,21
4,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/32/772447/boreal_ht_2020_202412131734106345_0000010.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/32/772447/boreal_ht_2020_202412131734106345_0000010.tif,boreal_ht_2020_202412131734106345_0000010.tif,10
5,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/34/885473/boreal_ht_2020_202412131734106348_0000043.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/34/885473/boreal_ht_2020_202412131734106348_0000043.tif,boreal_ht_2020_202412131734106348_0000043.tif,43
6,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/37/836853/boreal_ht_2020_202412131734106350_0000085.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/37/836853/boreal_ht_2020_202412131734106350_0000085.tif,boreal_ht_2020_202412131734106350_0000085.tif,85
7,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/38/642682/boreal_ht_2020_202412131734106352_0000016.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/38/642682/boreal_ht_2020_202412131734106352_0000016.tif,boreal_ht_2020_202412131734106352_0000016.tif,16
8,s3://maap-ops-workspace/aliz237/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/41/632171/boreal_ht_2020_202412131734106354_0000004.tif,/projects/my-private-bucket/dps_output/run_boreal_biomass_map/dev_v1.5/Ht_H30_2020/full_run_no_uncert/2024/12/13/08/12/41/632171/boreal_ht_2020_202412131734106354_0000004.tif,boreal_ht_2020_202412131734106354_0000004.tif,4


In [6]:
for(i in 1:3579){
  lvis_tile <- substring(RH098_list[i],first = 78,last = 104)
  
  ##Load lvis height map
  #rh098
  rh098_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("RH098"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_rh098 <- aws.s3::s3read_using(raster, object = rh098_path)  
  lvis_rh098 <- projectRaster(lvis_rh098,crs = "+proj=longlat +datum=WGS84 +no_defs")}
  
  #rh095
  rh095_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("RH095"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_rh095 <- aws.s3::s3read_using(raster, object = rh095_path) 
  lvis_rh095 <- projectRaster(lvis_rh095,crs = "+proj=longlat +datum=WGS84 +no_defs")}

  #rh090
  rh090_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("RH090"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_rh090 <- aws.s3::s3read_using(raster, object = rh090_path) 
  lvis_rh090 <- projectRaster(lvis_rh090,crs = "+proj=longlat +datum=WGS84 +no_defs")}

  #rh085
  rh085_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("RH085"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_rh085 <- aws.s3::s3read_using(raster, object = rh085_path) 
  lvis_rh085 <- projectRaster(lvis_rh085,crs = "+proj=longlat +datum=WGS84 +no_defs")}

  #rh080
  rh080_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("RH080"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_rh080 <- aws.s3::s3read_using(raster, object = rh080_path)  
  lvis_rh080 <- projectRaster(lvis_rh080,crs = "+proj=longlat +datum=WGS84 +no_defs")}

  
  #CC_gte_01p00m
  CC_gte_01p00m_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("CC_gte_01p00m"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_CC_gte_01p00m <- aws.s3::s3read_using(raster, object = CC_gte_01p00m_path) 
  lvis_CC_gte_01p00m <- projectRaster(lvis_CC_gte_01p00m,crs = "+proj=longlat +datum=WGS84 +no_defs")}

  #CC_gte_01p37m
  CC_gte_01p38m_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("CC_gte_01p37m"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_CC_gte_01p37m <- aws.s3::s3read_using(raster, object = CC_gte_01p37m_path)   
  lvis_CC_gte_01p37m <- projectRaster(lvis_CC_gte_01p37m,crs = "+proj=longlat +datum=WGS84 +no_defs")}

  #CC_gte_02p00m
  CC_gte_02p00m_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("CC_gte_02p00m"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_CC_gte_02p00m <- aws.s3::s3read_using(raster, object = CC_gte_02p00m_path) 
  lvis_CC_gte_02p00m <- projectRaster(lvis_CC_gte_02p00m,crs = "+proj=longlat +datum=WGS84 +no_defs")}
 
  #ZG_mean
  ZG_mean_path <- paste("s3://",lvis_path$X0[intersect(grep(paste0("ZG_mean"), lvis_path$X0), grep(paste0(lvis_tile), lvis_path$X0))],sep = "")
  lvis_ZG_mean <- aws.s3::s3read_using(raster, object = ZG_mean_path) 
  lvis_ZG_mean <- projectRaster(lvis_ZG_mean,crs = "+proj=longlat +datum=WGS84 +no_defs")}
  
  ##slope
  lvis_slope <-terrain(lvis_ZG_mean, opt=c('slope'), unit='degrees', neighbors=8) }

     
  ##Identify overlap between lvis and boreal tiles
  
    overlapping_check <- crop(boreal_tile,lvis_rh098)
  if(is.null(overlapping_check)==FALSE) {
    #tile <- overlapping_check@data$tile_num
    num_of_tiles <- length(overlapping_check@data$tile_num)
    for(j in 1:num_of_tiles) {
      #load icesat-2 ht map
      tile_number <- overlapping_check@data$tile_num[j]
      icesat2_ht_path <- ht_tindex_master$s3_path[which(ht_tindex_master$tile_num == tile_number)]
        icesat2_ht_map <-aws.s3::s3read_using(raster, object = icesat2_ht_path) 
        if(tryCatch(!is.null(crop(r,extent(icesat2_ht_map,extent(lvis_rh098)))), error=function(e) return(FALSE))==TRUE){
        icesat2_ht_map_subset <- crop(icesat2_ht_map,extent(lvis_rh098))
        icesat2_ht_map_subset_resample <- resample(x = icesat2_ht_map_subset,y = lvis_rh098)
        lvis_point <- data.frame(rasterToPoints(lvis_rh098))
        coordinates(lvis_point) <-~ x+y
        lvis_point@data$predicted_ht <- extract(icesat2_ht_map_subset_resample,lvis_point) 
        lvis_point@data$rh095 <- extract(lvis_rh095,lvis_point) 
        lvis_point@data$rh090 <- extract(lvis_rh090,lvis_point)  
        lvis_point@data$rh085 <- extract(lvis_rh085,lvis_point)  
        lvis_point@data$rh080 <- extract(lvis_rh080,lvis_point)  
        lvis_point@data$CC_gte_01p00m <- extract(lvis_CC_gte_01p00m,lvis_point)  
        lvis_point@data$CC_gte_01p37m <- extract(lvis_CC_gte_01p37m,lvis_point)  
        lvis_point@data$CC_gte_02p00m <- extract(lvis_CC_gte_01p00m,lvis_point)  
        lvis_point@data$slope <- extract(lvis_slope,lvis_point)
        lvis_point@data$year <- substring(gridded_lvis_rh098_name_list[1],first = 13,last = 16)
        lvis_point@data$Lon <- lvis_point@coords[,1]
        lvis_point@data$Lat <- lvis_point@coords[,2]
        lvis_point <- lvis_point@data[-which(is.na(lvis_point@data$predicted_ht)==TRUE),]
        names(lvis_point) <- c("RH098_mean_30m","Predicted_ht","RH095_mean_30m","RH090_mean_30m","RH085_mean_30m","RH080_mean_30m",
                                         "CC_gte_01p00m_30m","CC_gte_01p37m_30m","CC_gte_02p00m_30m","Slope",
                                         "Year","Lon","Lat")
        #reference_point <- reference_point_temp
          if(nrow(lvis_point) > 0 & j == 1) {
              ht_table <- lvis_point
             # z <- z + 1
          } 
          if(nrow(lvis_point) > 0 & j >1) {
              ht_table <- rbind(ht_table,lvis_point)
          }
      }
      
    }
    
    
  }
  print(i)
  print(nrow(ht_table))
  file_name <- paste("/projects/ht_val/",lvis_tile,".csv",sep = "")
  write.csv(ht_table,file_name)
  
}

[1] 2166


ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'print': error in evaluating the argument 'x' in selecting a method for function 'nrow': object 'ht_table' not found


In [14]:
for(j in 1:num_of_tiles) {
      #load icesat-2 ht map
      tile_number <- overlapping_check@data$tile_num[j]
      icesat2_ht_path <- paste("/projects/icesat2_gridded_ht_V6/tile_",overlapping_check@data$tile_num[j],".tif",sep = "")
      if(isTRUE(file.exists(icesat2_ht_path))==TRUE){
        icesat2_ht_map <- raster(icesat2_ht_path)
        if(tryCatch(!is.null(crop(r,extent(icesat2_ht_map,extent(lvis_rh098)))), error=function(e) return(FALSE))==TRUE){
        icesat2_ht_map_subset <- crop(icesat2_ht_map,extent(lvis_rh098))
        icesat2_ht_map_subset_resample <- resample(x = icesat2_ht_map_subset,y = lvis_rh098)
        lvis_point <- data.frame(rasterToPoints(lvis_rh098))
        coordinates(lvis_point) <-~ x+y
        lvis_point@data$predicted_ht <- extract(icesat2_ht_map_subset_resample,lvis_point) 
        lvis_point@data$rh095 <- extract(lvis_rh095,lvis_point) 
        lvis_point@data$rh090 <- extract(lvis_rh090,lvis_point)  
        lvis_point@data$rh085 <- extract(lvis_rh085,lvis_point)  
        lvis_point@data$rh080 <- extract(lvis_rh080,lvis_point)  
        lvis_point@data$CC_gte_01p00m <- extract(lvis_CC_gte_01p00m,lvis_point)  
        lvis_point@data$CC_gte_01p37m <- extract(lvis_CC_gte_01p37m,lvis_point)  
        lvis_point@data$CC_gte_02p00m <- extract(lvis_CC_gte_01p00m,lvis_point)  
        lvis_point@data$slope <- extract(lvis_slope,lvis_point)
        lvis_point@data$year <- substring(gridded_lvis_rh098_name_list[1],first = 13,last = 16)
        lvis_point@data$Lon <- lvis_point@coords[,1]
        lvis_point@data$Lat <- lvis_point@coords[,2]
        lvis_point <- lvis_point@data[-which(is.na(lvis_point@data$predicted_ht)==TRUE),]
        names(lvis_point) <- c("RH098_mean_30m","Predicted_ht","RH095_mean_30m","RH090_mean_30m","RH085_mean_30m","RH080_mean_30m",
                                         "CC_gte_01p00m_30m","CC_gte_01p37m_30m","CC_gte_02p00m_30m","Slope",
                                         "Year","Lon","Lat")
        #reference_point <- reference_point_temp
          if(nrow(lvis_point) > 0 & j == 1) {
              ht_table <- lvis_point
             # z <- z + 1
          } 
          if(nrow(lvis_point) > 0 & j >1) {
              ht_table <- rbind(ht_table,lvis_point)
          }
      }}
      
    }

In [18]:
icesat2_ht_map_subset <- crop(icesat2_ht_map,extent(lvis_rh098))
        icesat2_ht_map_subset_resample <- resample(x = icesat2_ht_map_subset,y = lvis_rh098)
        lvis_point <- data.frame(rasterToPoints(lvis_rh098))

ERROR: Error in .local(x, y, ...): extents do not overlap


In [19]:
tryCatch(!is.null(crop(r,extent(icesat2_ht_map,extent(lvis_rh098)))), error=function(e) return(FALSE))

[1] FALSE